# VGG16 pretrained model

In [ ]:
vgg_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
vgg_model.eval()

In [ ]:
print(vgg_model)

In [ ]:
print(vgg_model.features)

In [ ]:
for name, layer in vgg_model.named_children():
    print(f"Block: {name}")
    print(layer)
    print("-" * 20)

In [ ]:
for param in vgg_model.parameters():
    param.requires_grad = False

In [ ]:
vgg_model.classifier[6].in_features

In [ ]:
vgg_num_classes = 11
vgg_in_features = vgg_model.classifier[6].in_features

vgg_in_features

In [ ]:
vgg_model.classifier[6] = nn.Linear(vgg_in_features, vgg_num_classes)

In [ ]:
vgg_model.to(device)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    train_losses, val_losses, val_accuracies = [], [], []
    best_accuracy = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        model.eval()
        val_loss = 0.0
        correct, total = 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_losses.append(val_loss / len(val_loader))
        val_accuracy = correct / total
        val_accuracies.append(val_accuracy)

        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Val Accuracy: {val_accuracies[-1]:.4f}')

        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), 'best_vgg_model.pth')
            print("Model checkpoint saved")

    return train_losses, val_losses, val_accuracies

In [ ]:
def my_train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=10, save_path="best_model.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    mlflow.log_params({"epochs": epochs})
    
    train_acc_metric = Accuracy(task="multiclass", num_classes=11).to(device)
    val_acc_metric = Accuracy(task="multiclass", num_classes=11).to(device)

    train_loss_metric = MeanMetric().to(device)
    val_loss_metric = MeanMetric().to(device)

    history = {"train_loss":[], "val_loss":[], "val_acc":[]}
    best_accuracy = 0.0
    best_model_weights = copy.deepcopy(model.state_dict())

    train_bar = tqdm(range(epochs), desc="Total Progress", leave=True)
    for epoch in train_bar:
        print(f"Epoch {epoch + 1}/{epochs}")
        print("-" * 10)
    
        model.train()
        
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss_metric.update(loss)
            train_acc_metric.update(outputs, labels)

            train_bar.set_postfix(loss=loss.item())

        epoch_train_loss = train_loss_metric.compute()
        epoch_train_acc = train_acc_metric.compute()
        train_loss_metric.reset()
        train_acc_metric.reset()

        model.eval()
        
        val_bar = tqdm(val_loader, desc="Validation", leave=False)

        with torch.inference_mode():
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss_metric.update(loss)
                val_acc_metric.update(outputs, labels)

            epoch_val_loss = val_loss_metric.compute()
            epoch_val_acc = val_acc_metric.compute()
            val_loss_metric.reset()
            val_acc_metric.reset()

            history['train_loss'].append(epoch_train_loss.item())
            history['val_loss'].append(epoch_val_loss.item())
            history['val_acc'].append(epoch_val_acc.item())
            
            mlflow.log_metrics({
                "train_loss": epoch_train_loss.item(),
                "val_loss": epoch_val_loss.item(),
                "val_acc": epoch_val_acc.item()
                },
                step=epoch)

        
            if scheduler:
                scheduler.step()

            print(f'Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}')
 
            # --- Checkpointing ---
            if epoch_val_acc > best_accuracy:
                best_accuracy = epoch_val_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), save_path)
                print(f"--> New Best Model Saved (Acc: {best_accuracy:.4f})")

        print(f'Training complete. Best Validation Accuracy: {best_accuracy:.4f}')
        model.load_state_dict(best_model_wts)
        
    mlflow.pytorch.log_model(model, name="MLflow_tracked_VGG")
    
    return model, history

    

In [ ]:
optimizer = optim.Adam(vgg_model.parameters(),lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
criterion = nn.CrossEntropyLoss()


In [ ]:
train_losses, val_losses, val_accuracies = train_model(vgg_model, train_image_loader, test_image_loader, criterion, optimizer, epochs=20)

In [ ]:
summary(vgg_model, (3, 256, 256))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Loss plot
ax[0].plot(train_losses, label="Train Loss")
ax[0].plot(val_losses, label="Validation Loss")
ax[0].set_title("Loss Trend")
ax[0].legend()

# Accuracy plot
ax[1].plot(val_accuracies, label="Validation Accuracy")
ax[1].set_title("Accuracy Trend")
ax[1].legend()

plt.show()

In [ ]:
mlflow.set_experiment("Deep Learning Experiment")

In [ ]:
with mlflow.start_run():
    model, history = my_train_model(vgg_model, train_image_loader, test_image_loader, criterion, optimizer, scheduler, epochs=2, save_path="best_custom_model.pth")

In [ ]:
history